In [57]:
import altair as alt
import duckdb
import polars as pl

# Set up connection and test query
conn = duckdb.connect("../../data/ff_platform.duckdb",  read_only=True)

# Exploratory Section
First things first, let's determine how we might filter out some of the noise. I'd like to understand what records we should target to remove. Let's understand the low end of targets. 

In [18]:
conn.sql("""
      SELECT
          targets,
          COUNT(*) as games,
          ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) as pct_of_games
      FROM core.fct_player_game_stats
      WHERE position = 'WR'
      GROUP BY targets
      ORDER BY targets
  """).pl()

targets,games,pct_of_games
i32,i64,f64
0,19053,30.9
1,6392,10.4
2,5250,8.5
3,4754,7.7
4,4550,7.4
…,…,…
20,21,0.0
21,11,0.0
22,4,0.0


**Ehhhh** &rarr; this is - fine - but not super helpful. This removes even the best players worst weeks - like the best players will have bad weeks. And I want to see that story. Let's spin up another version of this query. this time broken out by season. 

In [19]:
receiving_games = conn.sql("""
      WITH player_season_totals AS (
          SELECT
              player_id,
              season,
              SUM(targets) as season_targets,
              COUNT(*) as games_played
          FROM core.fct_player_game_stats
          WHERE position = 'WR'
            AND season >= 2009
          GROUP BY player_id, season
      ),
      qualified_player_seasons AS (
          SELECT player_id, season
          FROM player_season_totals
          WHERE season_targets >= 40  -- 2 to 3 targets per game
      )
      SELECT
          g.player_id,
          g.season,
          g.week,
          g.position,
          g.team,
          g.targets,
          g.target_share,
          g.receptions,
          g.receiving_yards,
          g.receiving_epa,
          g.wopr,
          g.offense_pct
      FROM core.fct_player_game_stats g
      INNER JOIN qualified_player_seasons q
          ON g.player_id = q.player_id
          AND g.season = q.season
      WHERE g.position = 'WR'
        AND g.season >= 2009
      ORDER BY g.player_id, g.season, g.week
  """).pl()

print(f"Shape: {receiving_games.shape}")
print(f"Unique player-seasons: {receiving_games.group_by(['player_id', 'season']).n_unique()}")

Shape: (25633, 12)
Unique player-seasons: shape: (1_725, 12)
┌────────────┬────────┬──────┬──────────┬───┬─────────────────┬───────────────┬──────┬─────────────┐
│ player_id  ┆ season ┆ week ┆ position ┆ … ┆ receiving_yards ┆ receiving_epa ┆ wopr ┆ offense_pct │
│ ---        ┆ ---    ┆ ---  ┆ ---      ┆   ┆ ---             ┆ ---           ┆ ---  ┆ ---         │
│ str        ┆ i32    ┆ u32  ┆ u32      ┆   ┆ u32             ┆ u32           ┆ u32  ┆ u32         │
╞════════════╪════════╪══════╪══════════╪═══╪═════════════════╪═══════════════╪══════╪═════════════╡
│ 00-0033282 ┆ 2018   ┆ 13   ┆ 1        ┆ … ┆ 12              ┆ 13            ┆ 13   ┆ 13          │
│ 00-0027944 ┆ 2021   ┆ 11   ┆ 1        ┆ … ┆ 11              ┆ 11            ┆ 11   ┆ 10          │
│ 00-0028052 ┆ 2015   ┆ 12   ┆ 1        ┆ … ┆ 12              ┆ 12            ┆ 12   ┆ 12          │
│ 00-0037261 ┆ 2023   ┆ 18   ┆ 1        ┆ … ┆ 16              ┆ 18            ┆ 18   ┆ 17          │
│ 00-0034272 ┆ 2022   ┆ 20   ┆

In [20]:
chart = (receiving_games
      .group_by('season')
      .agg(pl.col('player_id').n_unique().alias('player_count'))
      .sort('season')
      .plot.bar(x='season', y='player_count'))
chart

alt.Chart(...)

In [21]:
receiving_games = receiving_games.with_columns(
      (pl.int_range(pl.len()) + 1)
      .over(['player_id', 'season'])
      .alias('game_num')
  )

  # Check a random player
first_player = receiving_games['player_id'][0]
receiving_games.filter(pl.col('player_id') == first_player).head(10)

player_id,season,week,position,team,targets,target_share,receptions,receiving_yards,receiving_epa,wopr,offense_pct,game_num
str,i32,i32,str,str,i32,f64,i32,i32,f64,f64,f64,i64
"""00-0002099""",2009,1,"""WR""","""SF""",8,0.258065,4,74,5.481047,0.689152,null,1
"""00-0002099""",2009,2,"""WR""","""SF""",8,0.296296,4,35,1.736027,0.714236,null,2
"""00-0002099""",2009,3,"""WR""","""SF""",5,0.2,2,38,-1.44183,0.431884,null,3
"""00-0002099""",2009,4,"""WR""","""SF""",4,0.173913,3,20,-0.729086,0.286558,null,4
"""00-0002099""",2009,5,"""WR""","""SF""",4,0.105263,0,0,-2.836721,0.269895,null,5
"""00-0002099""",2009,7,"""WR""","""SF""",6,0.1875,2,23,-4.407301,0.477788,null,6
"""00-0002099""",2009,8,"""WR""","""SF""",8,0.25,4,51,1.306547,0.694027,null,7
"""00-0002099""",2009,9,"""WR""","""SF""",2,0.045455,1,3,-0.573507,0.111259,null,8
"""00-0002099""",2009,11,"""WR""","""SF""",4,0.125,1,20,-1.997784,0.324962,null,9


# Check out a Hot Streak


In [23]:
# SET POSITION VAR
TARGET_POSITION = 'WR'

# 4-week rolling target share (lagged)
receiving_games = (receiving_games
    .filter(pl.col('position')==TARGET_POSITION)
    .with_columns(
      pl.col('target_share')
      .shift(1)
      .rolling_mean(window_size=4, min_samples=4)
      .over(['player_id', 'season'])
      .alias('rolling_4g_target_share')
      )
    )

# For now, set upper and lower quartiles as hot / cold
hot_threshold = receiving_games['rolling_4g_target_share'].quantile(0.75)
cold_threshold = receiving_games['rolling_4g_target_share'].quantile(0.25)

print(f"Hot threshold (75th pctl): {hot_threshold:.3f}")
print(f"Cold threshold (25th pctl): {cold_threshold:.3f}")

Hot threshold (75th pctl): 0.233
Cold threshold (25th pctl): 0.130


In [24]:
receiving_games = receiving_games.with_columns(
      pl.when(pl.col('rolling_4g_target_share') >= hot_threshold)
        .then(pl.lit('hot'))
        .when(pl.col('rolling_4g_target_share') <= cold_threshold)
        .then(pl.lit('cold'))
        .otherwise(pl.lit('average'))
        .alias('streak_tier')
  )

In [30]:
receiving_games.group_by('streak_tier').agg([
      pl.col('rolling_4g_target_share').mean().alias('mean_rolling_4g'),
      pl.col('target_share').mean().alias('mean_current_game'),
      pl.len().alias('n_games')
  ]).sort('mean_rolling_4g', descending=True)

### Commenting this for to make sure I'm following the logic. This is...
# - grouping the streak tier (which is essentially - upper, lower & middle bounds )
# - calculating their rolling 4g average mean
# - calculating the non-rolling target share mean
# - over x amount og games

streak_tier,mean_rolling_4g,mean_current_game,n_games
str,f64,f64,u32
"""hot""",0.279223,0.252446,4684
"""average""",0.180471,0.180811,16265
"""cold""",0.095379,0.121495,4684


### Check for Understanding
So this is telling me that...
1. **Hot Streak &rarr;** these players tend to see their streaks dip by a little bit. Only ~2% diff. Which this contextually makes sense as these are likely starters. 
2. **Average &rarr;** players with average weeks... well... stay there typically. (So they stay within the mean range)
3. **Cold &rarr;** these players typically see tier values **increase** after they have an off week - by about ~3% pts. this tracks as even the best players are going to have very bad games throughout the season. 

**Claude correction &rarr;** don't forget about the framing that it's less about movement back towards the mean. but more about how much of their streak they tend to retain. This might make more sense as I relate this closely to fantasy points. But let's keep exploring this with individual players

In [60]:
### let's find players using the dim_players table
players = conn.sql("""
    select * 
    from core.dim_players
    limit 10;
    """)

### cool let's grab some studs / players of interest
players_of_interest = [
      'Jaxon Smith-Njigba',
      'Deebo Samuel Sr.',
      'Puka Nacua',
      "Ja'Marr Chase",
      'Emeka Egbuka'
  ]

target_players = conn.execute("""
      SELECT player_id, display_name
      FROM core.dim_players
      WHERE display_name = ANY($1)
  """, [players_of_interest]).pl()


target_players_ids = target_players.select(pl.col('player_id')).to_series().to_list()
target_players_ids

['00-0036900', '00-0040129', '00-0039075', '00-0035719', '00-0038543']

In [61]:
target_players

player_id,display_name
str,str
"""00-0036900""","""Ja'Marr Chase"""
"""00-0040129""","""Emeka Egbuka"""
"""00-0039075""","""Puka Nacua"""
"""00-0035719""","""Deebo Samuel Sr."""
"""00-0038543""","""Jaxon Smith-Njigba"""


In [74]:
player_data = receiving_games.filter((pl.col('player_id') == '00-0038543'), pl.col('season')==2025)

  # Reshape for Altair (need long format for multiple lines)
plot_data = player_data.select(['game_num', 'season', 'target_share', 'rolling_4g_target_share']).unpivot(
      index=['game_num', 'season'],
      on=['target_share', 'rolling_4g_target_share'],
      variable_name='metric',
      value_name='value'
  )


In [75]:
base = alt.Chart(plot_data).encode(
    x='game_num:O',
    y='value:Q'
)

main_lines = base.mark_line(point=True).encode(
    color='metric:N'
)

mean_line = alt.Chart(plot_data).mark_rule(color='gray', strokeDash=[4,4]).encode(
    y=alt.datum(0.165)
)

alt.layer(main_lines, mean_line, data=plot_data).facet(
    row='season:N'
).resolve_scale(y='independent')

alt.FacetChart(...)